# EL-GHALI MOHAMED

### Modèle DBSCAN (Clustering Non Supervisé)

In [5]:
import pandas as pd
import math

#  NOTRE  DONNÉES Coordonnées X et Y de plusieurs points
data = {
    'Point': ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'],
    'X': [1, 1.5, 1, 8, 8.5, 8, 15, 2, 9, 14],
    'Y': [1, 1, 1.5, 8, 8, 8.5, 15, 1.5, 9, 2]
}
df = pd.DataFrame(data)

print("=== Nos points spatiaux ===")
print(df)

=== Nos points spatiaux ===
  Point     X     Y
0     A   1.0   1.0
1     B   1.5   1.0
2     C   1.0   1.5
3     D   8.0   8.0
4     E   8.5   8.0
5     F   8.0   8.5
6     G  15.0  15.0
7     H   2.0   1.5
8     I   9.0   9.0
9     J  14.0   2.0


### Fonction de Distance Mathématique
Pour savoir si deux points sont proches, nous avons besoin de calculer la distance qui les sépare. Nous utilisons la Distance Euclidienne.

In [6]:
def distance_euclidienne(point1, point2):
    """Calcule la distance en ligne droite entre deux points (X, Y)."""
    diff_x = point1[0] - point2[0]
    diff_y = point1[1] - point2[1]

    # Formule : Racine_carrée( (x1 - x2)² + (y1 - y2)² )
    return math.sqrt(diff_x**2 + diff_y**2)

def trouver_voisins(donnees, index_point, epsilon):
    """
    Cherche tous les points qui se trouvent dans le cercle de rayon epsilon
    autour d'un point.
    """
    voisins = []
    point_cible = donnees[index_point]

    for i, autre_point in enumerate(donnees):
        # Si la distance est inférieure ou égale au rayon epsilon, c'est un voisin
        if distance_euclidienne(point_cible, autre_point) <= epsilon:
            voisins.append(i)

    return voisins

### L'Algorithme DBSCAN
L'algorithme explore chaque point. S'il trouve un point avec suffisamment de voisins, il crée un cluster. Ensuite, il va voir les voisins de ce point, et s'ils ont eux-mêmes beaucoup de voisins, le cluster s'agrandit

In [7]:
def executer_dbscan(df_points, epsilon, min_pts):
    """Exécute l'algorithme DBSCAN complet sur nos données."""

    # On extrait juste les coordonnées X et Y sous forme de liste de tuples
    coords = list(zip(df_points['X'], df_points['Y']))

    # Initialisation : 0 signifie "Non Visité"
    # Plus tard, un nombre > 0 sera l'ID du cluster, et -1 signifiera Bruit
    etiquettes = [0] * len(coords)

    id_cluster_actuel = 0

    # On passe sur chaque point de notre dataset
    for i in range(len(coords)):
        # Si le point a déjà été assigné à un cluster ou marqué comme bruit, on l'ignore
        if etiquettes[i] != 0:
            continue

        #  Cherche ses voisins
        voisins = trouver_voisins(coords, i, epsilon)

        #  Vérification de la densité Est-ce un point central ?
        if len(voisins) < min_pts:
            # Pas assez de voisins : on le marque temporairement comme Bruit (-1)
            etiquettes[i] = -1
        else:
            # Assez de voisins : On démarre un NOUVEAU cluster
            id_cluster_actuel += 1
            etiquettes[i] = id_cluster_actuel

            # Expansion du cluster
            # On va analyser tous ses voisins, et les voisins de ses voisins
            i_voisin = 0
            while i_voisin < len(voisins):
                index_voisin = voisins[i_voisin]

                # Si le voisin avait été vu comme du Bruit avant, il devient finalement
                # un point Frontière de notre cluster actuel.
                if etiquettes[index_voisin] == -1:
                    etiquettes[index_voisin] = id_cluster_actuel

                # Si le voisin n'a jamais été visité
                elif etiquettes[index_voisin] == 0:
                    etiquettes[index_voisin] = id_cluster_actuel

                    # On regarde les propres voisins de ce voisin
                    nouveaux_voisins = trouver_voisins(coords, index_voisin, epsilon)

                    # Si ce voisin est aussi un point très dense, on ajoute ses
                    # propres voisins à notre liste de points à explorer
                    if len(nouveaux_voisins) >= min_pts:
                        # On ajoute les nouveaux voisins à notre liste sans faire de doublons
                        for nv in nouveaux_voisins:
                            if nv not in voisins:
                                voisins.append(nv)

                i_voisin += 1 # On passe au voisin suivant dans notre liste d'expansion

    return etiquettes

### Lancement et Résultats

In [8]:
# Paramètres du modèle
EPSILON = 1.5
MIN_PTS = 3

# On lance notre algorithme
resultats_clusters = executer_dbscan(df, EPSILON, MIN_PTS)

# On ajoute les résultats dans notre DataFrame pour y voir clair
df['Cluster_DBSCAN'] = resultats_clusters

df['Cluster_DBSCAN'] = df['Cluster_DBSCAN'].apply(lambda x: 'Bruit ' if x == -1 else f"Cluster {x}")

print("\n=== RÉSULTATS DU CLUSTERING ===")
print(df)

# Petit résumé automatique
print("\n--- Analyse ---")
nb_clusters = len([c for c in df['Cluster_DBSCAN'].unique() if 'Cluster' in c])
nb_bruit = len(df[df['Cluster_DBSCAN'] == 'Bruit'])

print(f"L'algorithme a trouvé {nb_clusters} groupe(s) dense(s).")
print(f"L'algorithme a isolé {nb_bruit} point(s) considéré(s) comme du bruit .")


=== RÉSULTATS DU CLUSTERING ===
  Point     X     Y Cluster_DBSCAN
0     A   1.0   1.0      Cluster 1
1     B   1.5   1.0      Cluster 1
2     C   1.0   1.5      Cluster 1
3     D   8.0   8.0      Cluster 2
4     E   8.5   8.0      Cluster 2
5     F   8.0   8.5      Cluster 2
6     G  15.0  15.0         Bruit 
7     H   2.0   1.5      Cluster 1
8     I   9.0   9.0      Cluster 2
9     J  14.0   2.0         Bruit 

--- Analyse ---
L'algorithme a trouvé 2 groupe(s) dense(s).
L'algorithme a isolé 0 point(s) considéré(s) comme du bruit .
